In [ ]:
# script to figure out the original reaction before rmg did the decay recipe in rmgpy.rmg.decay

In [1]:
import os
import sys

import rmgpy.data.rmg
import rmgpy.species
import rmgpy.rmg.decay

sys.path.append('/projects/westgroup/harris.se/autoscience/reaction_calculator/database')
import database_fun

os.environ['BABEL_LIBDIR'] = "/home/harris.se/anaconda3/envs/rmg_env/lib/openbabel/3.1.0/"

Loading DFT database from /projects/westgroup/harris.se/autoscience/reaction_calculator/database


In [2]:
# reaction_index = 4752
reaction_index = 4732
# reaction_index = 4745
reaction_index = 4741

reaction_index = 4951
rmg_reaction = database_fun.index2reaction(reaction_index)

In [3]:
database = rmgpy.data.rmg.RMGDatabase()
decay_families = ['intra_H_migration', 'R_Addition_MultipleBond']  # Add to this if you find more examples

database.load(
    path = rmgpy.settings['database.directory'],
    thermo_libraries = ['primaryThermoLibrary'],
    transport_libraries = [],
    reaction_libraries = [],
    seed_mechanisms = [],
    kinetics_families = decay_families,
    kinetics_depositories = ['training'],
    #frequenciesLibraries = self.statmechLibraries,
    depository = False,
)
for family in database.kinetics.families:
    if not database.kinetics.families[family].auto_generated:
        database.kinetics.families[family].add_rules_from_training(thermo_database=database.thermo)
        database.kinetics.families[family].fill_rules_by_averaging_up(verbose=True)


/home/harris.se/rmg/RMG-Py/rmgpy/rmg/reactors.py:53: RuntimeWarning: Unable to import Julia dependencies, original error: Exception 'ArgumentError' occurred while calling julia code:
const PyCall = Base.require(Base.PkgId(Base.UUID("438e738f-606a-5dbb-bf0a-cddfbfd45ab0"), "PyCall"))
  warnings.warn("Unable to import Julia dependencies, original error: " + str(e), RuntimeWarning)
ThermoData(Tdata=([300,400,500,600,800,1000,1500],'K'), Cpdata=([60.2599,68.0494,74.7775,80.9311,90.8846,97.4337,105.393],'J/(mol*K)'), H298=(-477.191,'kJ/mol'), S298=(269.551,'J/(mol*K)'), Cp0=(33.2579,'J/(mol*K)'), CpInf=(103.931,'J/(mol*K)'), comment="""Thermo group additivity estimation: group(O2s-(Cds-Cd)(Cds-Cd)) + group(O2s-(Cds-O2d)H) + group(Cds-OdOsOs) + group(Li-OCOdO) + radical(OC=OOJ)""").
The thermo for this species is probably wrong! Setting CpInf = Cphigh for Entropy calculationat T = 2000.0 K...
ThermoData(Tdata=([300,400,500,600,800,1000,1500],'K'), Cpdata=([60.2599,68.0494,74.7775,80.9311,90.

In [4]:
mol_reaction = rmgpy.reaction.Reaction()
mol_reaction.reactants = [sp.molecule[0] for sp in rmg_reaction.reactants]
mol_reaction.products = [sp.molecule[0] for sp in rmg_reaction.products]

In [5]:
possible_reactions = []
for family in decay_families:
    possible_reactions += database.kinetics.families[family].generate_reactions([sp.molecule[0] for sp in rmg_reaction.reactants])

In [6]:
decayed_matches = []
for i in range(len(possible_reactions)):
    
    new_reaction = rmgpy.reaction.Reaction()
    new_reaction.reactants = rmg_reaction.reactants
    
    decayed_products = []
    for p in possible_reactions[i].products:
        decayed_products += rmgpy.rmg.decay.decay_species(rmgpy.species.Species(molecule=[p]))
    
    new_reaction.products = decayed_products
    if new_reaction.is_isomorphic(rmg_reaction):
        decayed_matches.append(possible_reactions[i])

In [7]:
for d in decayed_matches:
    print(d)

<Molecule "[O]OCCCCOO"> <=> <Molecule "OO[CH]CCCOO">
<Molecule "[O]OCCCCOO"> <=> <Molecule "OO[CH]CCCOO">
<Molecule "[O]OCCCCOO"> <=> <Molecule "OO[CH]CCCOO">
<Molecule "[O]OCCCCOO"> <=> <Molecule "OO[CH]CCCOO">


In [8]:
for d in decayed_matches:
    query_reaction = rmgpy.reaction.Reaction()
    query_reaction.reactants = rmg_reaction.reactants
    query_reaction.products = [rmgpy.species.Species(molecule=[m]) for m in d.products]
    
    print(d, database_fun.get_unique_reaction_index(query_reaction))

IndexError: Reaction not in database

In [13]:
sp = rmgpy.species.Species(smiles='OO[CH]CCCOO')

In [ ]:
database_fun.add_species_to_database([sp])

In [9]:
query_reaction

In [10]:
database_fun.add_reaction_to_database([query_reaction])

Loaded reaction database contains 19663 unique reactions
Looking for new reactions in mechanism...
Found 1 new reactions
Added the following new reactions to the database:
	[O]OCCCCOO <=> OO[CH]CCCOO
Saving new reaction database...
Reaction database now contains 19664 unique reactions


In [11]:
database_fun.get_unique_reaction_index(query_reaction)

19694

In [14]:
database_fun.get_unique_species_index(sp)

2440

In [15]:
database_fun.get_unique_species_index(query_reaction.reactants[0])

289

In [19]:
database_fun.get_unique_species_index(database_fun.index2reaction(4744).products[1])

IndexError: list index out of range